In [6]:
import requests
import json
from datetime import datetime, timedelta
from time import sleep
import time

In [ ]:
# === EXTRACIÓN HISTÓRICO DATOS DESDE 01-01-2026 HASTA 26-01-2026 ===

API_KEY = "a8a28ae9b66b4f6783503688ca05f98e"
base_url = f"https://api.rawg.io/api/games"

resultados = []
errores = []

# Calcular días desde 2016-01-01 hasta hoy
fecha_inicio = datetime(2016, 1, 1)
dias_total = (datetime.now() - fecha_inicio).days

print(f"Extrayendo datos desde {fecha_inicio.strftime('%Y-%m-%d')} hasta hoy")
print(f"Total de días a procesar: {dias_total}\n")

for i in range(dias_total):
    
    fecha = (datetime(2016, 1, 1) + timedelta(days = i)).strftime("%Y-%m-%d")
    
    params = {
        "key"       : API_KEY,
        "page_size" : 40,
        "dates"     : f"{fecha},{fecha}"
    }

    try: 
        # Hacer la petición
        response = requests.get(base_url, params = params)
            
        if response.status_code == 200:
            data = response.json()
            count = data["count"]
            juegos = data["results"]
            
            print(f"{fecha}: {count} juegos encontrados", end = "\t")
        
            resultados.extend(juegos)
        
        else:
        print(f"Error:{response.status_code} en {fecha}")
        errores.append({"fecha": fecha, "status": response.status_code})
        
        # Rate limiting 
        sleep(1)

        
     except requests.exceptions.Timeout:
        print(f"\nTimeout en {fecha}")
        errores.append({"fecha": fecha, "error": "timeout"})
        sleep(5)
            
    except Exception as e:
        print(f"\nError inesperado en {fecha}: {e}")
        errores.append({"fecha": fecha, "error": str(e)})

print(f"\n\nEXTRACCIÓN COMPLETADA")
print(f"Juegos extraídos: {len(resultados)}")
print(f"Errores encontrados: {len(errores)}")   
        
    

In [ ]:
# ==== GUARDAR EXTRACCIÓN HISTÓRICA ====

with open(file = "extraccion_historica.json", mode = "w", encoding = "utf-8") as file:
    json.dump(resultados, file, indent=2, ensure_ascii=False)

print(f"Datos guardados en {file}")

